In [0]:
%sql
-- Databricks notebook source
-- MAGIC %md
-- MAGIC # 🧭 Compass — "where am I, what exists, what is it"
-- MAGIC Run top to bottom any time you sit back down on a project after a break.

-- COMMAND ----------

-- Where am I right now, by default, if I don't specify anything?
SELECT current_catalog() AS catalog, current_schema() AS schema;

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ## The big one — everything, with its real type
-- MAGIC `information_schema` is metadata Databricks maintains automatically in every
-- MAGIC catalog. This tells you MANAGED (real table) vs VIEW vs EXTERNAL — which
-- MAGIC `SHOW TABLES` alone will NOT tell you.

-- COMMAND ----------

SELECT table_schema, table_name, table_type
FROM nyc_taxi.information_schema.tables
ORDER BY table_schema, table_type, table_name;

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ## Step by step, the manual way (building → rooms → drawers → closets)

-- COMMAND ----------

SHOW CATALOGS;
-- every building

-- COMMAND ----------

SHOW SCHEMAS IN nyc_taxi;
-- rooms inside one building

-- COMMAND ----------

SHOW TABLES IN nyc_taxi.gold;
-- everything in one room (tables + views, mixed together)

-- COMMAND ----------

SHOW VIEWS IN nyc_taxi.gold;
-- just the views, isolated

-- COMMAND ----------

SHOW VOLUMES IN nyc_taxi.bronze;
-- volumes are separate — never show up in SHOW TABLES

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ## Zoom into ONE object — what is it, and if it's a view, what does it actually do?

-- COMMAND ----------

DESCRIBE TABLE EXTENDED nyc_taxi.gold.dim_pickup_zone;
-- scroll for the "Type" row (TABLE / VIEW) and, for views, "View Text" —
-- the actual saved SELECT statement, useful when you've forgotten how it was built

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ## Bonus — Delta-specific: what actually happened to this table over time?
-- MAGIC Every write, merge, and optimize on a Delta table is logged. Useful when a
-- MAGIC row count looks wrong and you want to know what ran, and when.

-- COMMAND ----------

DESCRIBE HISTORY nyc_taxi.silver.trips_clean;

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ## Bonus — row counts across several tables at once
-- MAGIC Unity Catalog has no single built-in "count every table" command — this
-- MAGIC UNION ALL pattern is the standard workaround. Adjust the table list as needed.

-- COMMAND ----------

SELECT 'bronze.yellow_tripdata' AS table_name, COUNT(*) AS row_count FROM nyc_taxi.bronze.yellow_tripdata
UNION ALL
SELECT 'silver.trips_clean',       COUNT(*) FROM nyc_taxi.silver.trips_clean
UNION ALL
SELECT 'silver.trips_quarantine',  COUNT(*) FROM nyc_taxi.silver.trips_quarantine
UNION ALL
SELECT 'gold.fact_trips',          COUNT(*) FROM nyc_taxi.gold.fact_trips;